# DPS crystal prior — Colab runner

This notebook contains **no logic**. It clones the repo, installs deps, mounts Drive
and calls the same CLI scripts you run locally. All code lives in git so the local
CPU test loop stays meaningful — edit there, push, re-run here.

**Order matters.** Sections 5–7 are a ~1h cheap read that surfaces problems before
you commit many GPU-hours to sections 8–9. Don't skip ahead: a wiring or convention
error found after a full pretrain costs a day.

**Runtime → Change runtime type → GPU** before starting.


In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 1. Repo


In [ ]:
import os, glob
REPO   = '/content/diffusion-posterior-sampling'
URL    = 'https://github.com/dawidratynski/diffusion-posterior-sampling.git'
BRANCH = 'scratchpad'   # <- the branch holding this work; change when merged to main

if os.path.exists(REPO):
    !cd {REPO} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull
else:
    !git clone --branch {BRANCH} {URL} {REPO}
%cd {REPO}

# Confirm you actually have the new code, not an older branch.
!git rev-parse --abbrev-ref HEAD && git log --oneline -1
assert os.path.exists('scripts/train_diffusion.py'), \
    'Wrong branch: scripts/train_diffusion.py is missing.'


## 2. Dependencies

`torch` is deliberately **not** a project dependency (it lives in the `cpu` group of
`pyproject.toml`, used only for local CPU dev), so this installs the project's other
deps and leaves Colab's CUDA torch untouched.


In [ ]:
!pip install -q -e .

# Fail loudly rather than limping on with missing deps: a failed editable
# install is easy to miss in Colab's output and only bites much later.
import importlib
for m in ['numpy', 'scipy', 'skimage', 'PIL', 'yaml', 'matplotlib', 'tqdm']:
    importlib.import_module(m)
print('dependencies OK')


## 3. Data (local disk) and checkpoints (Drive)

**Checkpoints go to Drive** — Colab disconnects and `/content` is lost.
**Data goes to local disk** — training reads thousands of small PNGs per epoch
and mounted-Drive I/O would starve the GPU.

The archive is located by searching for the directory containing `real/train`,
so it works whether or not the zip has a top-level folder inside it.

Set `DRIVE` and `ZIP` to match your Drive layout.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE  = '/content/drive/MyDrive/Non-movables'   # <- your Drive folder
ZIP    = f'{DRIVE}/dataset.zip'                  # <- your archive
MODELS = f'{DRIVE}/models'      # checkpoints on Drive: sessions die
os.makedirs(MODELS, exist_ok=True)

# ---------------------------------------------------------------------------
# THE ONLY THING TO CHANGE WHEN THE UVCGAN WEIGHTS ARRIVE.
# Upload the uvcgan2 model DIRECTORY (weights plus the config needed to rebuild
# the architecture -- not a bare .pth) to Drive and point this at it. Every cell
# below adapts automatically: the two models that need weights join the
# comparison, and work already done is kept.
UVCGAN_PATH = f'{DRIVE}/uvcgan_model'
HAVE_UVCGAN = os.path.exists(UVCGAN_PATH)

# Which R->S model builds the round-trip measurement. Held FIXED across the
# models being compared, or the comparison is not like-for-like. Set to
# 'spectral' to run the analytic-R->S variant of the experiment as well.
RS_FRAMEWORK = 'uvcgan2' if HAVE_UVCGAN else 'spectral'

print('UVCGAN weights:', 'FOUND' if HAVE_UVCGAN else f'not yet at {UVCGAN_PATH}')
print('round-trip R->S model:', RS_FRAMEWORK)
# ---------------------------------------------------------------------------


def latest_ckpt(d):
    """Newest model_ema_*.pt in d.

    Training cells save every --save_every steps, so you can interrupt one
    at any point and still have a usable checkpoint; this picks it up.
    """
    c = sorted(glob.glob(f'{d}/model_ema_*.pt'))
    if not c:
        raise FileNotFoundError(
            f'No model_ema_*.pt in {d}. Let the training cell reach at least one --save_every interval before running this.')
    return c[-1]

# Data goes on LOCAL disk, not Drive: training reads ~11.6k small PNGs per
# epoch and mounted-Drive I/O on many small files starves the GPU.
EXTRACT = '/content/data_raw'
if not os.path.exists(EXTRACT):
    !mkdir -p {EXTRACT} && unzip -q '{ZIP}' -d {EXTRACT}

# Archives may or may not carry a top-level folder, so find the directory
# that actually contains real/train rather than assuming the depth.
matches = sorted(glob.glob(f'{EXTRACT}/**/real/train', recursive=True))
assert matches, f'no */real/train found under {EXTRACT}; check the archive'
DATA = os.path.dirname(os.path.dirname(matches[0]))
print('DATA   =', DATA)
print('MODELS =', MODELS)


def drive_usage():
    """Checkpoints are the only thing that grows. Watch this between stages."""
    !du -sh {MODELS}/* 2>/dev/null; df -h /content/drive | tail -1

!ls {DATA}/real/train | wc -l && ls {DATA}/synth/train | wc -l


## 4. Split hygiene — **run this, do not just dry-run it**

Several crops come from each source photo, so the split must be at *source*
level; splitting on filenames leaks siblings across train/val and makes the
memorisation check meaningless.

**The archive on Drive predates the fix**, so the copy you just unpacked still
leaks. Run the first cell to confirm, then the second to repair it. Idempotent
— safe to re-run, and it skips domains that are already clean.


In [ ]:
!python scripts/fix_split.py {DATA}/real {DATA}/synth --dry_run


In [ ]:
# Apply it (expect ~4.6k real files to move; synth is already clean).
!python scripts/fix_split.py {DATA}/real {DATA}/synth


## 5. Environment smoke (~1 min)

The full train → generate → evaluate path at tiny scale. Proves the environment,
data layout and configs work together before anything expensive. Takes ~6 min on
CPU locally; much less here.

It also asserts the exported EMA checkpoint beats chance — guarding a bug where the
training log looked healthy while the saved model was still random initialisation.


In [ ]:
!PY=python ./scripts/smoke_e2e.sh {DATA} /content/smoke


## 6. Short real-only run (~1h) — the cheap read

A first prior at the **real** 160px resolution, trained only long enough to produce
meaningful output. This is deliberately *not* the final model: it exists to answer
"does DPS through this operator do anything sensible" in an hour rather than a day.

Checkpoints land every 1000 steps, so **you can interrupt this cell whenever you
like** and still use the latest one. Watch the `it/s` in the log to judge how far to
let it go.


In [ ]:
!python scripts/train_diffusion.py \
    --model_config configs/crystal_model_config.yaml \
    --data_root {DATA}/real/train \
    --val_root {DATA}/real/val --val_every 500 \
    --out_dir {MODELS}/quick_real \
    --batch_size 16 --lr 1e-4 --train_steps 8000 \
    --save_every 1000 --amp --resume


## 7. Generate + evaluate from the quick run

`--input_mode roundtrip` builds the measurement as `y = G_RS(real image)` and
records the real image as ground truth. Here the **analytic** operator plays
both roles — it builds `y` and DPS inverts it — which is self-consistent and
needs no trained weights.

Those two roles are separate flags (`--rs_framework`, `--dps_framework`) because
Part 2 needs them to differ: every model there must see the *same* measurement
while inverting its own operator.


In [ ]:
CKPT = latest_ckpt(f'{MODELS}/quick_real'); print('using', CKPT)
!sed 's|^model_path:.*|model_path: {CKPT}|' configs/crystal_model_config.yaml > /content/model_cfg.yaml

# DPS costs a UNet forward AND backward per step, so 1000 steps x many
# samples runs to hours. Use 100 steps for the cheap read; it is enough to
# judge whether the output is sane. Part 2 uses a 250-step schedule.
!sed 's|^timestep_respacing:.*|timestep_respacing: 100|' \
    configs/crystal_diffusion_config.yaml > /content/diffusion_fast.yaml

# Time ONE sample before committing to a batch. The analytic operator is used
# here in BOTH roles (it builds the measurement and DPS inverts it), which is
# self-consistent and needs no trained weights -- see the docstring of
# scripts/generate_augmented.py on why those two roles are separate flags.
import time; t0 = time.time()
!python scripts/generate_augmented.py \
    --model_config /content/model_cfg.yaml \
    --diffusion_config /content/diffusion_fast.yaml \
    --task_config configs/crystal_cyclegan_config.yaml \
    --input_mode roundtrip --real_root {DATA}/real/val \
    --rs_framework spectral \
    --method dps --dps_framework spectral --label dps_analytic \
    --out_dir /content/results/timing \
    --samples_per_input 1 --limit 1
per_sample = time.time() - t0
print(f'\n~{per_sample:.0f}s per 100-step sample.')
print(f'32 refs x 4 variants  = {128*per_sample/60:.0f} min at 100 steps')
print(f'at the 250-step schedule Part 2 uses = {per_sample*2.5:.0f}s each')


In [ ]:
# Adjust --limit from the timing above if needed.
!python scripts/generate_augmented.py \
    --model_config /content/model_cfg.yaml \
    --diffusion_config /content/diffusion_fast.yaml \
    --task_config configs/crystal_cyclegan_config.yaml \
    --input_mode roundtrip --real_root {DATA}/real/val \
    --rs_framework spectral \
    --method dps --dps_framework spectral --label dps_analytic \
    --out_dir /content/results/quick \
    --samples_per_input 4 --limit 32


In [ ]:
!python scripts/evaluate.py \
    --result_dir /content/results/quick \
    --real_root {DATA}/real/val \
    --real_train_root {DATA}/real/train \
    --out_csv /content/results/quick_per_image.csv


### Look at the images, not only the table

The metrics cannot tell you whether the output is plausible to a human eye. Compare
a generated image against the reference it was derived from.


In [ ]:
import matplotlib.pyplot as plt, csv
rows = list(csv.DictReader(open('/content/results/quick/manifest.csv')))[:4]
fig, ax = plt.subplots(2, len(rows), figsize=(3*len(rows), 6))
for i, r in enumerate(rows):
    ax[0, i].imshow(plt.imread(r['source_image'])); ax[0, i].set_title('reference', fontsize=9)
    ax[1, i].imshow(plt.imread(f"/content/results/quick/generated/{r['output_image']}"))
    ax[1, i].set_title('DPS', fontsize=9)
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.show()


## 8. `scale` sweep on the quick prior (~75 min) — optional

Superseded by **section 13**, which sweeps every DPS variant against the *final*
prior and at the schedule actually used for results. Run this one only if you
want an early read while sections 9–10 are still pending; otherwise skip
straight to Part 2.

Note the optimum depends on both the prior and the step count, so a value tuned
here does not transfer to the final runs.


In [ ]:
SCALES = [0.3, 1.0, 2.0, 3.0]
N_REFS, N_VARIANTS = 24, 2

DISCARD_OLD = True   # earlier runs used a different operator; do not reuse them
if DISCARD_OLD:
    !rm -rf /content/results/scale_*

print(f'{len(SCALES)} scales x {N_REFS*N_VARIANTS} samples x 100 steps')
print(f'at ~23 s/sample that is ~{len(SCALES)*N_REFS*N_VARIANTS*23/60:.0f} min total')

for s in SCALES:
    out = f'/content/results/scale_{s}'
    if os.path.exists(f'{out}/manifest.csv'):
        print(f'scale {s}: already present, skipping')
        continue
    !python scripts/generate_augmented.py \
        --model_config /content/model_cfg.yaml \
        --diffusion_config /content/diffusion_fast.yaml \
        --task_config configs/crystal_cyclegan_config.yaml \
        --input_mode roundtrip --real_root {DATA}/real/val \
        --rs_framework spectral \
        --method dps --dps_framework spectral --label dps_analytic \
        --out_dir {out} --scale {s} \
        --samples_per_input {N_VARIANTS} --limit {N_REFS}

# Confirm the subsets really are spread across photos, not crops of one scene.
import csv, re
for s in SCALES:
    rows = list(csv.DictReader(open(f'/content/results/scale_{s}/manifest.csv')))
    photos = {re.sub(r'_sample_\d+\.png$', '', os.path.basename(r['source_image']))
              for r in rows}
    print(f'scale {s}: {len(rows)} images from {len(photos)} distinct photos')

dirs = ' '.join(f'--result_dir /content/results/scale_{s}' for s in SCALES)
!python scripts/evaluate.py {dirs} --real_root {DATA}/real/val


---
# Only continue once sections 5–7 look right.
Everything below is measured in GPU-hours.
---


## 9. Stage 1 — pretrain on SYNTH (~18 h, 2 sessions)

Synth is effectively unlimited, so this stage carries no memorisation risk and
teaches lattice structure and low-level statistics.

**40k steps, not the 150k originally planned.** At the measured 0.61 it/s,
150k + 40k would be ~87 hours — seven Colab sessions. Synth images are also
clean and highly structured, so they converge far faster than a natural-image
corpus; 40k is a realistic starting budget rather than a target.

**Judge it by the probe, not the step count.** Watch `val @ step ... t=25`:
when it stops falling, extend or stop accordingly. The training loss will have
flattened thousands of steps earlier and tells you nothing.

`--resume` means re-running this cell after a disconnect continues rather than
restarting, so spanning two sessions costs nothing.


In [ ]:
!python scripts/train_diffusion.py \
    --model_config configs/crystal_model_config.yaml \
    --data_root {DATA}/synth/train \
    --val_root {DATA}/synth/val --val_every 2000 \
    --out_dir {MODELS}/synth_pretrain \
    --batch_size 16 --lr 1e-4 --train_steps 40000 \
    --save_every 2000 --keep_last 1 --save_fp16 --amp --resume


### 9b. Free the pretrain optimiser state before section 10

`ckpt_latest.pt` is ~1.35 GB (model + EMA + both AdamW moments) and exists only
to resume *that* run. Section 10 starts a fresh run from the exported weights via
`--init_from`, so once section 9 is finished this file is dead weight — and on a
4 GB Drive it is the difference between fitting and not.

**Only run this once section 9 has actually finished.** Deleting it means section 9
can no longer be extended; you would have to restart it.


In [ ]:
drive_usage()

SYNTH_DONE = False   # <- set True only after section 9 reports 'Training finished.'
if SYNTH_DONE:
    !rm -f {MODELS}/synth_pretrain/ckpt_latest.pt
    print('freed ~1.35 GB')
    drive_usage()
else:
    print('set SYNTH_DONE = True once section 9 has finished')


## 10. Stage 2 — finetune on REAL (~4.5 h)

`--init_from` starts from the pretrained weights with a fresh optimiser and
step counter (unlike `--resume`, which continues a run). Lower LR, because the
point is to move the prior to the real domain, not retrain it.

**You now have two stopping signals, and they point in opposite directions:**

| signal | says | source |
|---|---|---|
| `val t=25` still falling | keep training | the probe, logged here |
| `NN dist to real train` approaching 1.11 | stop, it is copying | section 11 |
| `peak prominence` still above real | keep training | section 11 |

Real is the scarce side (~2.3k source photos), so this stage is where
memorisation happens. Re-run section 11 against successive checkpoints. Stop
where prominence has come down to the real value but NN distance has not yet
fallen to the floor.

If no such window exists — prominence never reaches real before NN distance
collapses — that is a genuine finding about the real set being too small, not a
failure. Report it as such.


In [ ]:
PRETRAINED = latest_ckpt(f'{MODELS}/synth_pretrain'); print('init from', PRETRAINED)
!python scripts/train_diffusion.py \
    --model_config configs/crystal_model_config.yaml \
    --data_root {DATA}/real/train \
    --val_root {DATA}/real/val --val_every 1000 \
    --out_dir {MODELS}/real_finetune \
    --init_from {PRETRAINED} \
    --batch_size 16 --lr 2e-5 --train_steps 10000 \
    --save_every 1250 --keep_last 8 --save_fp16 --amp --resume


---
# Part 2 — the three-model comparison

Everything below produces the report. It compares three synth→real models:

| model | what it is | needs UVCGAN weights |
|---|---|---|
| `uvcgan` | direct S→R translation | yes |
| `dps_uvcgan` | DPS inverting the **trained** R→S generator | yes |
| `dps_analytic` | DPS inverting the **analytic** operator | no |

**These cells need no editing when the weights arrive.** Set `UVCGAN_PATH` in
section 3, re-run that cell, then re-run these. The two gated models join the
comparison and completed work is kept. Until then they are skipped and
`dps_analytic` runs alone — which still validates the pipeline, the metrics and
the figures end to end.

Two experiments run for every model:

- **roundtrip** — real → `G_RS` → model. The original real image is ground
  truth, so PSNR/SSIM/LPIPS apply.
- **synth** — synthetic image → model. The actual deliverable. No ground truth
  (given a synthetic input there is no single correct realistic image), so
  distributional and lattice metrics only.

**Two caveats to carry into the writeup.** The round trip structurally favours
`uvcgan`, because `gen_ab` was trained with a cycle loss to invert `gen_ba` — a
DPS win there is strong, a loss is not damning. And `dps_analytic` receives the
same measurement as the others while inverting an operator that did not produce
it; that mismatch is not a flaw in the experiment, it is what the ablation
measures.


## 12. Probe the UVCGAN checkpoint — run FIRST when weights arrive

`load_uvcgan2` was written against the uvcgan2 API without a real checkpoint to
test against, so it is the least-verified piece of the whole pipeline. The probe
reports what is actually in the directory, then walks the exact path the
experiments take — load both generators, check shapes and ranges, check the DPS
gradient survives — and says what to fix if a step fails.

It also checks that `G_RS` actually lands in the synth domain, which catches a
reversed direction convention (is domain `a` synth in your uvcgan2 config?). A
generator can load, run and differentiate perfectly while mapping the wrong way.

Ten seconds here, versus three GPU-hours to discover the same thing.


In [ ]:
# Install uvcgan2 itself. The loader needs its code to rebuild the generator
# architecture from the saved config -- the checkpoint alone is not enough.
#
# --no-deps is deliberate: uvcgan2's requirements pin torch versions, and letting
# pip resolve them can replace Colab's CUDA torch with a CPU build, which fails
# much later and confusingly. Everything it actually needs is already present;
# if an import below reports a genuinely missing module, install that one alone.

if not HAVE_UVCGAN:
    print('No weights yet -- skipping uvcgan2 install.')
else:
    UVCGAN_SRC = '/content/uvcgan2'
    if not os.path.exists(UVCGAN_SRC):
        !git clone -q https://github.com/LS4GAN/uvcgan2.git {UVCGAN_SRC}
    !pip install -q -e {UVCGAN_SRC} --no-deps

    import importlib
    try:
        importlib.import_module('uvcgan2')
        import torch
        print('uvcgan2 imported OK | torch', torch.__version__,
              '| cuda', torch.cuda.is_available())
        assert torch.cuda.is_available(), \
            'torch lost CUDA -- a dependency replaced it; reinstall the GPU build'
    except ImportError as e:
        print(f'uvcgan2 import failed: {e}\n'
              'Install just that missing module, then re-run this cell.')


In [ ]:
# === 12. Probe the UVCGAN checkpoint — run FIRST when weights arrive ========
# load_uvcgan2 was written against the uvcgan2 API without a real checkpoint to
# test against, so it is the least-verified piece of the pipeline. This reports
# what is actually in the directory, then walks the exact path the experiments
# take (load both generators, check ranges, check the DPS gradient survives) and
# says what to fix if a step fails. Ten seconds here, versus three GPU-hours to
# discover the same thing.

if not HAVE_UVCGAN:
    print(f'No weights at {UVCGAN_PATH}.\n'
          'Set UVCGAN_PATH in section 3 when they arrive, re-run that cell, '
          'then this one.\nEverything below still runs with dps_analytic alone.')
else:
    !python scripts/probe_uvcgan.py --path {UVCGAN_PATH} \
        --real_root {DATA}/real/val --synth_root {DATA}/synth/val


## 13. Hyperparameter sweep — conditioning scale, per DPS variant

The two DPS variants invert **different operators**, so they need separate
optima: a scale tuned against one says nothing about the other. This sweeps each
over the same grid and writes a full metric table per variant, which is what
justifies the chosen value in the report rather than asserting it.

**Reading the table — no single column decides it:**

- **label validity** (`spacing_err_median`, `signature_err_median`) is decisive.
  Drift means mislabelled data, which is worse than no data.
- **`prominence_median`** should land *near* the real value printed by the
  evaluator, not as high or as low as possible.
- **`diversity_rmse`** likewise has a sensible range rather than a direction.
  Two crops of the *same* real photo differ by ≈0.088 RMSE and two *different*
  real photos by ≈0.144, so far above that is over-dispersion, not richer
  realism.
- **`nn_dist_median`** below the printed floor means the prior is copying
  training images.
- **`kid`** is the distributional metric to trust at this sample size; `fid` is
  reported for familiarity and flagged when its covariance estimate is
  meaningless.

Rule of thumb: the largest scale that has not yet damaged diversity.


In [ ]:
SWEEP_SCALES = '0.3,1.0,2.0,3.0'
SWEEP_REFS, SWEEP_VARIANTS = 24, 2

CKPT = latest_ckpt(f'{MODELS}/real_finetune'); print('prior:', CKPT)
!sed 's|^model_path:.*|model_path: {CKPT}|' configs/crystal_model_config.yaml > /content/model_cfg.yaml

# One schedule shared by the sweep and section 14, so the tuned scale is the
# scale that actually gets used. DPS applies one guidance step per timestep, so
# total guidance scales with schedule length: a value tuned at 100 steps
# over-guides at 250.
!sed 's|^timestep_respacing:.*|timestep_respacing: 250|' \
    configs/crystal_diffusion_config.yaml > /content/diffusion_eval.yaml

n_dps = 2 if HAVE_UVCGAN else 1
n_scales = len(SWEEP_SCALES.split(','))
print(f'~{n_dps * n_scales * SWEEP_REFS * SWEEP_VARIANTS * 29 / 60:.0f} min '
      f'at ~29 s/sample ({n_dps} DPS variant(s) x {n_scales} scales)')

!python scripts/run_comparison.py --stage sweep \
    --data_root {DATA} \
    --model_config /content/model_cfg.yaml \
    --diffusion_config /content/diffusion_eval.yaml \
    --task_config configs/crystal_cyclegan_config.yaml \
    --out_dir /content/results/sweep \
    --uvcgan_path "{UVCGAN_PATH if HAVE_UVCGAN else ''}" \
    --rs_framework {RS_FRAMEWORK} \
    --scales {SWEEP_SCALES} \
    --sweep_limit {SWEEP_REFS} --sweep_samples {SWEEP_VARIANTS}


In [ ]:
# The sweep tables, as they will appear in the report.
import glob

import pandas as pd

COLS = ['model', 'scale', 'spacing_err_median', 'spacing_err_p90',
        'spacing_err_strong', 'signature_err_median', 'psnr_median',
        'ssim_median', 'lpips_median', 'prominence_median',
        'spectrum_dist_to_real', 'kid', 'diversity_rmse', 'nn_dist_median']

for f in sorted(glob.glob('/content/results/sweep/*_sweep.csv')):
    df = pd.read_csv(f)
    print(f'\n=== {f.split("/")[-1]} ===')
    print(df[[c for c in COLS if c in df.columns]].to_string(index=False))


In [ ]:
# Chosen from the tables above. These are SEPARATE decisions: the two variants
# invert different operators, so do not copy one value to the other.
SCALE_DPS_ANALYTIC = 2.0   # <- from the dps_analytic table
SCALE_DPS_UVCGAN   = 1.0   # <- from the dps_uvcgan table (unused until weights exist)
print(f'analytic {SCALE_DPS_ANALYTIC}   uvcgan {SCALE_DPS_UVCGAN}')


## 14. Final comparison — both experiments, all available models

Generates, evaluates and builds the figures in one pass. Resumable: a re-run
after the weights arrive redoes only the two missing models.

**Budget.** Each DPS model costs `limit × samples × 250 steps`. At 50 references
× 4 variants that is 200 samples ≈ 1.6 h per DPS model per experiment; `uvcgan`
is a single forward pass and essentially free. Lower `FINAL_LIMIT` if units are
tight — the metrics stabilise well before 50 references.

With weights present: 2 DPS models × 2 experiments ≈ 6.5 h. Without: ≈ 3.2 h.


In [ ]:
FINAL_LIMIT, FINAL_SAMPLES = 50, 4

# Rebuilt here rather than inherited from section 13, so this cell also works if
# you skipped the sweep and reused previously chosen scales. Idempotent.
CKPT = latest_ckpt(f'{MODELS}/real_finetune'); print('prior:', CKPT)
!sed 's|^model_path:.*|model_path: {CKPT}|' configs/crystal_model_config.yaml > /content/model_cfg.yaml
!sed 's|^timestep_respacing:.*|timestep_respacing: 250|' \
    configs/crystal_diffusion_config.yaml > /content/diffusion_eval.yaml

n_dps = 2 if HAVE_UVCGAN else 1
print(f'~{2 * n_dps * FINAL_LIMIT * FINAL_SAMPLES * 29 / 3600:.1f} h '
      f'for {n_dps} DPS model(s) x 2 experiments')

!python scripts/run_comparison.py --stage final \
    --data_root {DATA} \
    --model_config /content/model_cfg.yaml \
    --diffusion_config /content/diffusion_eval.yaml \
    --task_config configs/crystal_cyclegan_config.yaml \
    --out_dir /content/results/final \
    --uvcgan_path "{UVCGAN_PATH if HAVE_UVCGAN else ''}" \
    --rs_framework {RS_FRAMEWORK} \
    --scale_dps_analytic {SCALE_DPS_ANALYTIC} \
    --scale_dps_uvcgan {SCALE_DPS_UVCGAN} \
    --limit {FINAL_LIMIT} --samples {FINAL_SAMPLES}


In [ ]:
# The two comparison tables the report is built on.
import pandas as pd

COLS = ['model', 'scale', 'spacing_err_median', 'spacing_err_strong',
        'spacing_err_p90', 'signature_err_median', 'angle_err_median',
        'psnr_median', 'ssim_median', 'lpips_median',
        'prominence_median', 'src_prominence_median', 'spectrum_dist_to_real',
        'kid', 'kid_std', 'fid', 'fid_reliable',
        'diversity_rmse', 'nn_dist_median']

for mode in ('roundtrip', 'synth'):
    f = f'/content/results/final/{mode}_metrics.csv'
    if not os.path.exists(f):
        print(f'{mode}: not run yet'); continue
    df = pd.read_csv(f)
    print(f'\n=== {mode} ===')
    print(df[[c for c in COLS if c in df.columns]].to_string(index=False))


In [ ]:
# Look at the figures before trusting the tables. Aggregates cannot tell you
# whether the output is plausible to a human eye; twice in this project the eye
# caught something the metrics only articulated afterwards.
import glob

from IPython.display import Image as ShowImage
from IPython.display import display

for mode in ('roundtrip', 'synth'):
    for f in sorted(glob.glob(
            f'/content/results/final/figures/{mode}/grids/comparison_01.png')):
        print(f'=== {mode}: reference | measurement | each model ===')
        display(ShowImage(filename=f))

# And the stochasticity that a deterministic translator cannot offer.
for f in sorted(glob.glob(
        '/content/results/final/figures/roundtrip/grids/variants_*_01.png')):
    print(f'=== {f.split("/")[-1]} ===')
    display(ShowImage(filename=f))


## 15. Copy everything to Drive

`/content` is lost when the session ends, and this is hours of GPU output.

```
results_<timestamp>/
  roundtrip_metrics.csv     <- report table 1
  synth_metrics.csv         <- report table 2
  *_per_image.csv           <- per-image values, for error bars or histograms
  figures/<mode>/grids/     <- comparison_NN.png, variants_<model>_NN.png
  figures/<mode>/panels/    <- every cell as its own file, for hand layout
  figures/<mode>/index.csv  <- panel dir -> source image
  sweep/*_sweep.csv         <- hyperparameter tables
```

`docs/methods.md` in the repo explains every metric and the analytic operator,
with formulas, for writing this up.


In [ ]:
import datetime

stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
dest = f'{DRIVE}/results_{stamp}'
!mkdir -p {dest}
!cp -r /content/results/final/. {dest}/ 2>/dev/null
!mkdir -p {dest}/sweep && cp /content/results/sweep/*_sweep.csv {dest}/sweep/ 2>/dev/null

print('saved to', dest)
!du -sh {dest}
!df -h /content/drive | tail -1
print('\nRemember to disconnect the runtime if you are done '
      '— an idle GPU still burns compute units.')
